In [6]:
import pandas as pd
import os
import json
import numpy as np
from mplsoccer import Pitch ,VerticalPitch
pd.set_option('display.max_columns', None)


In [7]:
def time_to_seconds(time_str):
    if time_str is None or (isinstance(time_str, float) and np.isnan(time_str)):
        return 90 * 60
    h, m, s = map(int, str(time_str).split(':'))
    return h * 3600 + m * 60 + s

In [8]:
match_ids = [e.name for e in os.scandir('data/matches') if e.is_dir()]

In [9]:
import os
import json
import numpy as np
import pandas as pd


FRAME_RATE         = 10
MAX_DIST_PER_FRAME = 1.5


def time_to_seconds(time_str):
    if time_str is None or (isinstance(time_str, float) and np.isnan(time_str)):
        return 90 * 60
    h, m, s = map(int, str(time_str).split(':'))
    return h * 3600 + m * 60 + s


def pull_data(match_ids):
    '''
    Builds one big dataframe for all matches following the SkillCorner tutorial.

    For each event in dynamic_events, captures every player's position at
    frame_end — exactly like tutorial section 5.1, but for all events and
    all matches combined.

    Adds runner, ball_carrier, tip flags per the tutorial.
    Adds velocity, speed, and acceleration columns.

    input:  match_ids (list of int)
    output: full_df (pd.DataFrame)
    '''
    all_synced = []

    for match in match_ids:

        # ── Step 2: Load tracking data ─────────────────────────────────────────
        raw_data = pd.read_json(
            f'data/matches/{match}/{match}_tracking_extrapolated.jsonl', lines=True
        )
        raw_df = pd.json_normalize(
            raw_data.to_dict('records'),
            record_path='player_data',
            meta=['frame', 'timestamp', 'period', 'possession', 'ball_data']
        )
        raw_df['possession_player_id'] = raw_df['possession'].apply(lambda x: x.get('player_id'))
        raw_df['possession_group']     = raw_df['possession'].apply(lambda x: x.get('group'))
        raw_df[['ball_x', 'ball_y', 'ball_z', 'is_detected_ball']] = pd.json_normalize(raw_df['ball_data'])
        raw_df = raw_df.drop(columns=['possession', 'ball_data'])
        raw_df['match_id'] = match

        # ── Step 3: Load match metadata ────────────────────────────────────────
        with open(f'data/matches/{match}/{match}_match.json', encoding='utf-8') as f:
            raw_match_data = json.load(f)

        raw_match_df = pd.json_normalize(raw_match_data, max_level=2)
        raw_match_df['home_team_side'] = raw_match_df['home_team_side'].astype(str)

        players_df = pd.json_normalize(
            raw_match_df.to_dict('records'),
            record_path='players',
            meta=[
                'home_team_score', 'away_team_score', 'date_time', 'home_team_side',
                'home_team.name', 'home_team.id', 'away_team.name', 'away_team.id',
            ]
        )

        players_df = players_df[~(players_df['start_time'].isna() & players_df['end_time'].isna())]
        players_df['total_time']       = players_df['end_time'].apply(time_to_seconds) - players_df['start_time'].apply(time_to_seconds)
        players_df['is_gk']            = players_df['player_role.acronym'] == 'GK'
        players_df['match_name']       = players_df['home_team.name'] + ' vs ' + players_df['away_team.name']
        players_df['home_away_player'] = np.where(players_df['team_id'] == players_df['home_team.id'], 'Home', 'Away')
        players_df['team_name']        = np.where(
            players_df['team_id'] == players_df['home_team.id'],
            players_df['home_team.name'], players_df['away_team.name']
        )
        players_df[['home_team_side_1st_half', 'home_team_side_2nd_half']] = (
            players_df['home_team_side']
            .astype(str).str.strip('[]').str.replace("'", '', regex=False)
            .str.split(', ', expand=True)
        )
        players_df['direction_player_1st_half'] = np.where(
            players_df['home_away_player'] == 'Home',
            players_df['home_team_side_1st_half'], players_df['home_team_side_2nd_half']
        )
        players_df['direction_player_2nd_half'] = np.where(
            players_df['home_away_player'] == 'Home',
            players_df['home_team_side_2nd_half'], players_df['home_team_side_1st_half']
        )
        players_df = players_df[[
            'match_name', 'home_team.name', 'away_team.name', 'id', 'short_name', 'number',
            'team_id', 'team_name', 'player_role.position_group', 'total_time',
            'player_role.name', 'player_role.acronym', 'is_gk',
            'direction_player_1st_half', 'direction_player_2nd_half'
        ]]

        # ── Step 4: Merge tracking + metadata (tutorial style) ────────────────
        enriched_tracking_data = raw_df.merge(players_df, left_on='player_id', right_on='id')

        # ── Step 5: Load dynamic events ────────────────────────────────────────
        de_match = pd.read_csv(f'data/matches/{match}/{match}_dynamic_events.csv')
        de_match = de_match[[
            'player_id', 'match_id', 'frame_start', 'frame_end', 'event_id',
            'event_type', 'team_id', 'player_in_possession_id', 'team_in_possession_phase_type',
            'lead_to_shot', 'lead_to_goal', 'one_touch', 'is_header', 'game_interruption_after',
            'team_in_possession_phase_type', 'player_targeted_dangerous', 'end_type', 'start_type',
            'pass_outcome', 'xthreat', 'xpass_completion'
        ]].rename(columns={
            'player_id': 'player_id_event',
            'team_id':   'team_id_event',
        })

        # ── Step 5.1: Expand events across all frames between frame_start and frame_end
        # Instead of just the snapshot at frame_end, we get every tracking frame
        # that falls within the event's duration — one row per player per frame per event.
        de_match_expanded = []
        for _, event in de_match.iterrows():
            event_frames = enriched_tracking_data[
                (enriched_tracking_data['frame'] >= event['frame_start']) &
                (enriched_tracking_data['frame'] <= event['frame_end'])
            ].copy()
            for col in de_match.columns:
                event_frames[col] = event[col].values[0] if hasattr(event[col], 'values') else event[col]
            de_match_expanded.append(event_frames)

        synced = pd.concat(de_match_expanded, ignore_index=True)

        # Tutorial flags
        synced['runner']       = synced['player_id_event'] == synced['player_id']
        synced['ball_carrier'] = synced['player_in_possession_id'] == synced['player_id']
        synced['tip']          = synced['team_id_event'] == synced['team_id']

        all_synced.append(synced)
        print(f'✓ match {match}: {len(synced)} rows, {synced["event_id"].nunique()} events')

    full_df = pd.concat(all_synced, ignore_index=True)

    # ── Velocity, Speed & Acceleration ────────────────────────────────────────
    full_df = full_df.sort_values(['player_id', 'frame']).reset_index(drop=False)
    full_df = full_df.drop_duplicates(subset=['player_id', 'frame', 'event_id'], keep='first')

    full_df['x_smooth'] = full_df.groupby('player_id')['x'].transform(
        lambda s: s.rolling(5, min_periods=1, center=True).mean()
    )
    full_df['y_smooth'] = full_df.groupby('player_id')['y'].transform(
        lambda s: s.rolling(5, min_periods=1, center=True).mean()
    )
    full_df['frame_diff'] = (
        full_df.groupby('player_id')['frame']
        .diff().fillna(1).replace(0, 1).astype(float)
    )

    full_df['vx'] = full_df.groupby('player_id')['x_smooth'].diff().astype(float) / full_df['frame_diff']
    full_df['vy'] = full_df.groupby('player_id')['y_smooth'].diff().astype(float) / full_df['frame_diff']

    jump_mask = np.sqrt(full_df['vx'].astype(float)**2 + full_df['vy'].astype(float)**2) > MAX_DIST_PER_FRAME
    full_df.loc[jump_mask, ['vx', 'vy']] = np.nan

    full_df['ax'] = full_df.groupby('player_id')['vx'].diff().astype(float) / full_df['frame_diff']
    full_df['ay'] = full_df.groupby('player_id')['vy'].diff().astype(float) / full_df['frame_diff']

    for col in ['vx', 'vy', 'ax', 'ay']:
        full_df[col] = full_df[col].astype(float).fillna(0)

    full_df['vx_kmh']    = full_df['vx'] * FRAME_RATE * 3.6
    full_df['vy_kmh']    = full_df['vy'] * FRAME_RATE * 3.6
    full_df['speed_kmh'] = np.sqrt(full_df['vx_kmh'].astype(float)**2 + full_df['vy_kmh'].astype(float)**2)
    full_df['ax_kmh2']   = full_df['ax'] * FRAME_RATE * 3.6
    full_df['ay_kmh2']   = full_df['ay'] * FRAME_RATE * 3.6

    full_df = full_df.drop(columns=['x_smooth', 'y_smooth', 'frame_diff', 'vx', 'vy', 'ax', 'ay'])
    full_df = full_df.sort_values('index').reset_index(drop=True)
    full_df = full_df.drop(columns=['index'])

    return full_df

In [10]:
df = pull_data(match_ids)


✓ match 1886347: 1603228 rows, 5079 events
✓ match 2015213: 1409276 rows, 4582 events
✓ match 2013725: 1486078 rows, 4999 events
✓ match 2006229: 1534258 rows, 4991 events
✓ match 2011166: 1332562 rows, 3966 events
✓ match 1996435: 1628088 rows, 5292 events


/var/folders/rr/7hqln5dd7_55vxvrgjhmz1gw0000gn/T/ipykernel_94367/3281878309.py:100: DtypeWarning: Columns (0: game_interruption_before, 1: game_interruption_after, 2: first_line_break_type, 3: pressing_chain_end_type) have mixed types. Specify dtype option on import or set low_memory=False.
  de_match = pd.read_csv(f'data/matches/{match}/{match}_dynamic_events.csv')


✓ match 2017461: 1345388 rows, 4188 events
✓ match 1925299: 1685464 rows, 5220 events
✓ match 1953632: 1464738 rows, 4823 events


/var/folders/rr/7hqln5dd7_55vxvrgjhmz1gw0000gn/T/ipykernel_94367/3281878309.py:100: DtypeWarning: Columns (0: affected_line_breaking_passing_option_run_subtype) have mixed types. Specify dtype option on import or set low_memory=False.
  de_match = pd.read_csv(f'data/matches/{match}/{match}_dynamic_events.csv')


✓ match 1899585: 1415414 rows, 4713 events


In [11]:
df.iloc[418:440]

,x,y,player_id,is_detected,frame,timestamp,period,possession_player_id,possession_group,ball_x,ball_y,ball_z,is_detected_ball,match_id,match_name,home_team.name,away_team.name,id,short_name,number,team_id,team_name,player_role.position_group,total_time,player_role.name,player_role.acronym,is_gk,direction_player_1st_half,direction_player_2nd_half,player_id_event,frame_start,frame_end,event_id,event_type,team_id_event,player_in_possession_id,team_in_possession_phase_type,lead_to_shot,lead_to_goal,one_touch,is_header,game_interruption_after,player_targeted_dangerous,end_type,start_type,pass_outcome,xthreat,xpass_completion,runner,ball_carrier,tip,vx_kmh,vy_kmh,speed_kmh,ax_kmh2,ay_kmh2
418,-40.39,0.20,51009,False,49,2026-04-12 00:00:03.900,1.0,51649.0,away team,-21.38,0.95,0.26,True,1886347,Auckland FC vs Newcastle United Jets FC,Auckland FC,Newcastle United Jets FC,51009,R. Scott,1,1805,Newcastle United Jets FC,Other,5400,Goalkeeper,GK,True,left_to_right,right_to_left,735578,48,58,7_1,passing_option,1805,51649.0,create,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0001,0.9865,False,False,True,-0.072,0.216,0.227684,-2.557954e-13,0.000000e+00
419,-21.53,-10.07,176224,True,49,2026-04-12 00:00:03.900,1.0,51649.0,away team,-21.38,0.95,0.26,True,1886347,Auckland FC vs Newcastle United Jets FC,Auckland FC,Newcastle United Jets FC,176224,P. Cancar,4,1805,Newcastle United Jets FC,Central Defender,5400,Right Center Back,RCB,False,left_to_right,right_to_left,735578,48,58,7_1,passing_option,1805,51649.0,create,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0001,0.9865,False,False,True,-1.368,-0.360,1.414576,-1.278977e-13,0.000000e+00
420,-22.30,1.33,51649,True,49,2026-04-12 00:00:03.900,1.0,51649.0,away team,-21.38,0.95,0.26,True,1886347,Auckland FC vs Newcastle United Jets FC,Auckland FC,Newcastle United Jets FC,51649,A. Šušnjar,15,1805,Newcastle United Jets FC,Central Defender,5400,Left Center Back,LCB,False,left_to_right,right_to_left,735578,48,58,7_1,passing_option,1805,51649.0,create,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0001,0.9865,False,True,True,0.144,1.584,1.590532,1.278977e-13,0.000000e+00
421,4.61,-32.23,50983,True,49,2026-04-12 00:00:03.900,1.0,51649.0,away team,-21.38,0.95,0.26,True,1886347,Auckland FC vs Newcastle United Jets FC,Auckland FC,Newcastle United Jets FC,50983,D. Ingham,14,1805,Newcastle United Jets FC,Full Back,5400,Right Back,RB,False,left_to_right,right_to_left,735578,48,58,7_1,passing_option,1805,51649.0,create,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0001,0.9865,False,False,True,5.400,0.792,5.457771,0.000000e+00,-5.115908e-13
422,-20.69,16.74,735578,True,49,2026-04-12 00:00:03.900,1.0,51649.0,away team,-21.38,0.95,0.26,True,1886347,Auckland FC vs Newcastle United Jets FC,Auckland FC,Newcastle United Jets FC,735578,M. Natta,33,1805,Newcastle United Jets FC,Full Back,5400,Left Back,LB,False,left_to_right,right_to_left,735578,48,58,7_1,passing_option,1805,51649.0,create,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0001,0.9865,True,False,True,0.072,1.224,1.226116,1.278977e-13,-1.278977e-13
423,-4.58,7.37,50978,True,49,2026-04-12 00:00:03.900,1.0,51649.0,away team,-21.38,0.95,0.26,True,1886347,Auckland FC vs Newcastle United Jets FC,Auckland FC,Newcastle United Jets FC,50978,C. Timmins,19,1805,Newcastle United Jets FC,Midfield,5098,Left Defensive Midfield,LDM,False,left_to_right,right_to_left,735578,48,58,7_1,passing_option,1805,51649.0,create,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0001,0.9865,False,False,True,0.648,-0.792,1.023312,-6.394885e-14,0.000000e+00
424,-10.62,-2.52,735574,True,49,2026-04-12 00:00:03.900,1.0,51649.0,away team,-21.38,0.95,0.26,True,1886347,Auckland FC vs Newcastle United Jets FC,Auckland FC,Newcastle United Jets FC,735574,K. Grozos,17,1805,Newcastle United Jets FC,Midfield,5400,Right Defensive Midfield,RDM,False,left_to_right,right_to_left,735578,48,58,7_1,passing_option,1805,51649.0,create,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0001,0.9865,False,False,True,-2.304,3.816,4.457608,-6.394885e-14

In [12]:
df.to_csv("data/full_data.csv")